# CatBoost Training

CatBoost or Categorical Boosting was developed by Yandex.

It is an ML algorithm based on gradient-boosted decision trees. It supports categorical values natively, handles null values automatically, and uses ordered boosting to prevent data leakage.

## Import Libraries

Import the required libraries.

In [ ]:
# Import required libraries

import json
import warnings

import numpy as np
import pandas as pd
import optuna
from optuna.pruners import MedianPruner

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

## Load Processed Data

Load the pre-processed data.

In [ ]:
# Load Dataset

train = pd.read_parquet(
    "/kaggle/input/datasets/shivamgravity/pgs-s6e6-processed-data-v1/train_processed.parquet"
)
test = pd.read_parquet(
    "/kaggle/input/datasets/shivamgravity/pgs-s6e6-processed-data-v1/test_processed.parquet"
)
sample_submission = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv"
)


## Configs

Define config values to use later.

In [ ]:
# CONFIGS

N_SPLITS = 3
RANDOM_STATE = 42
N_TRIALS = 30

ID = "id"
TARGET = "class"

# ====================================
# Splitting Features and Target values
# ====================================

X = train.drop([ID,TARGET],axis=1)
y = train[TARGET]

# =========================
# Features
# =========================

FEATURES = X.columns.tolist()

# =========================
# Categorical Features
# =========================

cat_features = [
    "spectral_type",
    "galaxy_population"
]

cat_feature_indices = [
    X.columns.get_loc(col)
    for col in cat_features
]

## Optuna Optimzation

Optimizing the Catboost Model using Optuna hyperparameter tuning.

In [ ]:
# ==========================================
# Optuna Objective
# ==========================================

def objective(trial):

    params = {
        "loss_function": "MultiClass",
        "random_seed": 42,
        "verbose": 0,
        "task_type": "GPU",

        "iterations": trial.suggest_int(
            "iterations",
            500,
            3000
        ),

        "depth": trial.suggest_int(
            "depth",
            4,
            10
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1,
            20
        ),

        "random_strength": trial.suggest_float(
            "random_strength",
            0,
            10
        ),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0,
            10
        ),

        "border_count": trial.suggest_int(
            "border_count",
            32,
            255
        )
    }

    early_stopping = trial.suggest_int("early_stopping_rounds", 20, 300)

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = []

    for train_idx, valid_idx in cv.split(X, y):

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            cat_features=cat_feature_indices,
            eval_set=(X_valid, y_valid),
            early_stopping_rounds=early_stopping,
            use_best_model=True
        )

        preds = model.predict(X_valid).flatten()

        score = balanced_accuracy_score(
            y_valid,
            preds
        )

        scores.append(score)
        
        trial.report(
            np.mean(scores),
            step=len(scores)
        )
        
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)

In [ ]:
# ==========================================
# Run Study
# ==========================================

study = optuna.create_study(
    direction="maximize",
    study_name="catboost_s6e6",
    pruner=MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=1
    )
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

## CV Results

Cross Validation results with the best params.

In [ ]:
# ==========================================
# Best Results
# ==========================================

best_score = study.best_value
best_params = study.best_params

print("\n" + "=" * 60)
print("BEST CV SCORE")
print(best_score)

print("\nBEST PARAMETERS")
print(best_params)
print("=" * 60)

In [ ]:
# Save params

with open("best_catboost_params.json", "w") as f:
    json.dump(best_params, f, indent=4)

results_df = pd.DataFrame({
    "best_cv_score": [best_score]
})

results_df.to_csv(
    "best_cv_score.csv",
    index=False
)

## Train Final Model

Training the final_model with best params after hyperparameter optimization using optuna.

In [ ]:
# ==========================================
# Train Final Model
# ==========================================

final_params = {
    **best_params,
    "loss_function": "MultiClass",
    "random_seed": 42,
    "verbose": 200,
    "task_type": "GPU"
}

final_model = CatBoostClassifier(
    **final_params
)

final_model.fit(
    X,
    y,
    cat_features=cat_feature_indices
)

## Predict the Test Data

Predicting the Test data using the Final trained model with best params.

In [ ]:
# ==========================================
# Predict Test
# ==========================================

X_test = test[FEATURES]

test_preds = final_model.predict(
    X_test
).flatten()


## Saving the Result

Saving the predicted result for competition submission.

In [ ]:
submission = pd.DataFrame({
    "id": test[ID],
    "class": test_preds
})

submission.to_csv(
    "submission.csv",
    index=False
)

print("Submission saved.")